In [9]:
import cv2
import numpy as np
import mediapipe as mp
import time
import json


def find_red_markers(frame_bgr):
    """
    BGR 이미지를 입력받아,
    HSV 마스크로 빨간색 영역을 찾아
    각 덩어리(마커)의 중심 좌표 리스트를 반환.
    """
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)

    # 🎯 빨간색 범위 (조금 넉넉하게 설정)
    # Hue:   0~15, 160~180
    # Sat/Val 최소값을 50으로 내려서 연한 빨강/주황/핑크도 잡히게
    lower_red1 = np.array([0,   120, 80])
    upper_red1 = np.array([10,  255, 255])
    lower_red2 = np.array([170, 120, 80])
    upper_red2 = np.array([180, 255, 255])


    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = cv2.bitwise_or(mask1, mask2)

    # 잡음 제거용 morphological operation
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 컨투어(빨간 영역) 찾기
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    red_points = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        # 🔧 마커가 작게 찍혀 있어도 잡히도록 threshold 낮춤
        if area < 20:  # 필요하면 3~10 사이로 조절
            continue
        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        red_points.append((cx, cy))

    return red_points, mask


def in_box(pt, box):
    """pt = (x, y), box = (x_min, y_min, x_max, y_max)"""
    x, y = pt
    x_min, y_min, x_max, y_max = box
    return (x_min <= x <= x_max) and (y_min <= y <= y_max)


def main():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ 웹캠을 열 수 없습니다.")
        return

    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    # 왼/오 볼 윤곽을 이루는 랜드마크 인덱스
    # 왼쪽 기준으로, 오른쪽은 대칭되는 인덱스로 튜닝
    left_cheek_region =  [234,  93, 132,  58, 172, 136, 150, 176, 148, 152]
    right_cheek_region = [454, 323, 361, 288, 397, 365, 379, 400, 377, 152]

    # 볼 영역 bounding box 확장 margin (픽셀 단위)
    margin_x = 25   # 좌우로 확장
    margin_y = 25   # 상하로 확장

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("❌ 프레임을 읽을 수 없습니다.")
                break

            h, w, _ = frame.shape

            # 1) 얼굴 랜드마크 (볼 polygon 계산용)
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_image)

            left_poly_np = None
            right_poly_np = None
            left_box = None
            right_box = None

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # ---------------------------
                # 왼쪽 볼 polygon + bounding box
                # ---------------------------
                left_poly = []
                for idx in left_cheek_region:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    left_poly.append((u, v))
                    # 윤곽 점 (빨강)
                    cv2.circle(frame, (u, v), 3, (0, 0, 255), -1)

                if len(left_poly) >= 3:
                    left_poly_np = np.array(left_poly, np.int32)
                    cv2.polylines(
                        frame,
                        [left_poly_np],
                        isClosed=True,
                        color=(0, 0, 255),
                        thickness=1,
                    )

                    # bounding box 계산 + margin 확장
                    lx_min = min(p[0] for p in left_poly) - margin_x
                    lx_max = max(p[0] for p in left_poly) + margin_x
                    ly_min = min(p[1] for p in left_poly) - margin_y
                    ly_max = max(p[1] for p in left_poly) + margin_y

                    # 화면 경계 내로 클램핑
                    lx_min = max(lx_min, 0)
                    ly_min = max(ly_min, 0)
                    lx_max = min(lx_max, w - 1)
                    ly_max = min(ly_max, h - 1)

                    left_box = (lx_min, ly_min, lx_max, ly_max)

                    # 디버깅용: bounding box 사각형 표시 (연한 빨강)
                    cv2.rectangle(
                        frame,
                        (lx_min, ly_min),
                        (lx_max, ly_max),
                        (0, 0, 150),
                        1,
                    )

                # ---------------------------
                # 오른쪽 볼 polygon + bounding box
                # ---------------------------
                right_poly = []
                for idx in right_cheek_region:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    right_poly.append((u, v))
                    # 윤곽 점 (파랑)
                    cv2.circle(frame, (u, v), 3, (255, 0, 0), -1)

                if len(right_poly) >= 3:
                    right_poly_np = np.array(right_poly, np.int32)
                    cv2.polylines(
                        frame,
                        [right_poly_np],
                        isClosed=True,
                        color=(255, 0, 0),
                        thickness=1,
                    )

                    rx_min = min(p[0] for p in right_poly) - margin_x
                    rx_max = max(p[0] for p in right_poly) + margin_x
                    ry_min = min(p[1] for p in right_poly) - margin_y
                    ry_max = max(p[1] for p in right_poly) + margin_y

                    rx_min = max(rx_min, 0)
                    ry_min = max(ry_min, 0)
                    rx_max = min(rx_max, w - 1)
                    ry_max = min(ry_max, h - 1)

                    right_box = (rx_min, ry_min, rx_max, ry_max)

                    # bounding box 사각형 표시 (연한 파랑)
                    cv2.rectangle(
                        frame,
                        (rx_min, ry_min),
                        (rx_max, ry_max),
                        (150, 0, 0),
                        1,
                    )

            # 2) HSV 마스크로 빨간 마커 찾기
            red_points, red_mask = find_red_markers(frame)

            left_markers = []
            right_markers = []
            others = []

            # 3) 각 빨간 점이 어느 볼 박스 안에 들어가는지 검사
            for (cx, cy) in red_points:
                in_left = False
                in_right = False

                if left_box is not None and in_box((cx, cy), left_box):
                    in_left = True

                if right_box is not None and in_box((cx, cy), right_box):
                    in_right = True

                if in_left and not in_right:
                    left_markers.append((cx, cy))
                    # 왼쪽 볼 위의 빨간 마커: 연두색 크게
                    cv2.circle(frame, (cx, cy), 7, (0, 255, 0), -1)
                    cv2.putText(
                        frame,
                        "L",
                        (cx + 5, cy - 5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (0, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )
                elif in_right and not in_left:
                    right_markers.append((cx, cy))
                    # 오른쪽 볼 위의 빨간 마커: 하늘색 크게
                    cv2.circle(frame, (cx, cy), 7, (255, 255, 0), -1)
                    cv2.putText(
                        frame,
                        "R",
                        (cx + 5, cy - 5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )
                else:
                    others.append((cx, cy))
                    # 얼굴 밖/볼 밖의 빨간 점: 보라색
                    cv2.circle(frame, (cx, cy), 6, (255, 0, 255), -1)

            # 4) 디버깅/로깅용 JSON (원하면 주석 처리해도 됨)
            payload = {
                "timestamp": time.time(),
                "left_markers": [{"u": int(u), "v": int(v)} for (u, v) in left_markers],
                "right_markers": [{"u": int(u), "v": int(v)} for (u, v) in right_markers],
                "others": [{"u": int(u), "v": int(v)} for (u, v) in others],
            }
            print(json.dumps(payload), flush=True)

            # 5) 화면 표시
            cv2.imshow("Face + Cheek Boxes + Red Markers", frame)
            # 빨간 검출 마스크도 같이 보고 싶으면 아래 줄 주석 해제
            # cv2.imshow("Red Mask", red_mask)  # (수정해야함)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        face_mesh.close()
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


{"timestamp": 1764127379.3925078, "left_markers": [], "right_markers": [], "others": [{"u": 599, "v": 448}, {"u": 197, "v": 443}, {"u": 601, "v": 428}, {"u": 617, "v": 412}, {"u": 231, "v": 339}]}
{"timestamp": 1764127379.4338443, "left_markers": [], "right_markers": [], "others": [{"u": 196, "v": 443}, {"u": 599, "v": 429}, {"u": 609, "v": 405}, {"u": 230, "v": 339}, {"u": 234, "v": 324}]}
{"timestamp": 1764127379.4453738, "left_markers": [], "right_markers": [], "others": [{"u": 193, "v": 445}, {"u": 596, "v": 425}, {"u": 608, "v": 406}, {"u": 590, "v": 400}, {"u": 592, "v": 388}, {"u": 227, "v": 338}, {"u": 236, "v": 322}]}
{"timestamp": 1764127379.4653432, "left_markers": [], "right_markers": [], "others": [{"u": 609, "v": 462}, {"u": 597, "v": 449}, {"u": 589, "v": 392}, {"u": 224, "v": 340}, {"u": 234, "v": 323}, {"u": 174, "v": 280}, {"u": 250, "v": 273}]}
{"timestamp": 1764127379.482773, "left_markers": [], "right_markers": [], "others": [{"u": 600, "v": 395}, {"u": 582, "v": 3

KeyboardInterrupt: 